# Northwoodfresh Data Analysis and Summary Statistics

Module 1 Project Exercise

Author: A. Chris Yi
Date: 2026-10-05


## Imports

In [41]:
import pandas as pd
import numpy as np

northwoodfresh_data = pd.read_csv("northwoodfresh_data.csv")
northwoodfresh_data.head(13)


,region,month,revenue
0,Northeast,1,532914.57
1,Northeast,2,516405.13
2,Northeast,3,536839.90
3,Northeast,4,559598.78
4,Northeast,5,513912.01
5,Northeast,6,513912.44
6,Northeast,7,561059.53
7,Northeast,8,539953.30
8,Northeast,9,507793.67
9,Northeast,10,534106.56


Data is imported, checks good. Used a head of 13 to make sure that there's more than one region shown. The decimal place rounding worked properly, and the output is numeric so calculations can be run.

## Summary Statistics by Region

In [27]:
region_stats = (
    northwoodfresh_data
    .groupby("region")["revenue"]
    .agg(
        mean="mean",
        median="median",
        std="std",
        min="min",
        max="max",
        var="var",
        q1=lambda x: x.quantile(0.25),
        q3=lambda x: x.quantile(0.75)
    )
    .reset_index()
    .sort_values("region")
)

region_stats["range"] = (
    region_stats["max"] - region_stats["min"]
)

region_stats["iqr"] = (
    region_stats["q3"] - region_stats["q1"]
)

print(region_stats)

      region           mean     median           std        min        max  \
0    Midwest  494923.588333  491253.53  25492.221372  461211.35  547374.04   
1  Northeast  527694.838333  524659.85  19350.267268  507793.67  561059.53   
2  Southeast  420460.297500  428321.65  56127.372840  347414.39  489016.75   
3       West  580188.961667  583852.91  26790.433030  532189.73  621185.11   

            var           q1           q3      range          iqr  
0  6.498534e+08  473906.9250  512833.9875   86162.69   38927.0625  
1  3.744328e+08  512421.7925  537618.2500   53265.86   25196.4575  
2  3.150282e+09  367029.8325  468450.9450  141602.36  101421.1125  
3  7.177273e+08  564278.0775  595895.9275   88995.38   31617.8500  


Statistical summaries by group using .groupby aggregation. Follow the row numbers for the var, q1, q3, range, and iqr on the second section of the table. 

## Distribution Visualizations

### Distribution Visualization Southeast Region

In [40]:
import plotly.express as px

southeast_data = northwoodfresh_data[
    northwoodfresh_data["region"] == "Southeast"
]

fig = px.histogram(
    southeast_data,
    y="revenue",
    x="month",
    nbins=30,
    title="Revenue Distribution - Southeast",
)
fig.update_layout(
    title_x=0.5
)
fig.update_yaxes(title_text="Revenue ($)")
fig.update_xaxes(title_text="Month")
fig.show()

The southwest region shows a marked dropdown after month 6. It appears to be statistically significant and bimodal. 

### Distribution Visualization by Region

In [39]:
fig = px.box(
    northwoodfresh_data,
    x="region",
    y="revenue",
    title="Revenue Distribution by Region"
)
fig.update_layout(
    title_x=0.5
)
fig.update_yaxes(title_text="Revenue ($)")
fig.update_xaxes(title_text="Region")
fig.show()

Box plot of all regions revenue distribution. This agrees that the Southeast region is indeed an outlier and we need to address the Southeast question.

## The Southeast Question

Confidence interval for mean monthly revenue for the Southeast region, a 2-sample t-test, and effect size. 

In [42]:
from scipy import stats

southeast_months_1_6 = southeast_data[
    southeast_data["month"].between(1, 6)
]

southeast_months_7_12 = southeast_data[
    southeast_data["month"].between(7, 12)
]

print(f"Mean revenue for Southeast months 1-6: {southeast_months_1_6['revenue'].mean()}")
print(f"Mean revenue for Southeast months 7-12: {southeast_months_7_12['revenue'].mean()}")

t_stat, p_value = stats.ttest_ind(
    southeast_months_1_6['revenue'],
    southeast_months_7_12['revenue'],
    equal_var=False
)
print(f"T-statistic: {t_stat}, P-value: {p_value}")

Mean revenue for Southeast months 1-6: 471595.645
Mean revenue for Southeast months 7-12: 369324.95
T-statistic: 9.787951734507637, P-value: 8.80589611336903e-06


In [44]:
mean_diff = southeast_months_1_6["revenue"].mean() - southeast_months_7_12["revenue"].mean()

n1, n2 = len(southeast_months_1_6), len(southeast_months_7_12)
s1, s2 = southeast_months_1_6["revenue"].std(), southeast_months_7_12["revenue"].std()
se_diff = mean_diff / np.sqrt(s1**2/n1 + s2**2/n2)

margin = 1.96 * se_diff
ci_lower = mean_diff - margin
ci_upper = mean_diff + margin

pooled_std = np.sqrt(((n1 - 1) * s1**2 + (n2 - 1) * s2**2) / (n1 + n2 - 2))
cohens_d = mean_diff / pooled_std

print(f"Mean difference months 1-6 vs 7-12: {mean_diff}")
print(f"95% Confidence Interval for the difference: ({ci_lower}, {ci_upper})")
print(f"Cohen's d for the difference: {cohens_d}")


Mean difference months 1-6 vs 7-12: 102270.695
95% Confidence Interval for the difference: (102251.51061460037, 102289.87938539965)
Cohen's d for the difference: 5.651076568733049


As expected, the mean for months 1-6 is 471595.65 which is close to 480000. The mean for months 7-12 is 369324.95 which is close to 370000. The mean difference is 102270.70 which is close to 110000. The 95% CI doesn't include zero and the p-value from the t-test is miniscule. The Cohen's D substantially greater than 0.8 so the effect is indeed large. 